# Yorùbá RAG corpus

This notebook builds `dataset_rag.csv`, the knowledge base for the RAG. When a student asks Gemma 3:4b something in Yorùbá, the closest rows are retrieved and given to Gemma as context, so it answers from verified content instead of hallucinating.

**Run it top to bottom (Run All).** Every dataset cell adds rows to the same `rag_rows` list, so running one of them again on its own adds its rows twice.

## Output columns

| Column | What it holds |
|---|---|
| `language` | `yor` = Yorùbá with English,
| `source_dataset` | The dataset and split the row comes |
| `search_context` | The text that is embedded and searched: the question or sentence, as Yorùbá \| English |
| `pedagogical_answer` | The text given to Gemma as context when this row is retrieved |

## Datasets

| # | Dataset | Source | Splits used | Rows | What it gives Gemma |
|---|---|---|---|---|---|
| 1 | AfriMGSM | `masakhane/afrimgsm` (`yor`, `eng`) | train, test | 258 | Math word problems in Yorùbá and English with the final answer (step-by-step only in the 8 train rows) |
| 2 | AfriMMLU | `masakhane/afrimmlu` (`yor`, `eng`) | validation, dev, test | 608 | Multiple-choice questions in 5 school subjects, with the options and the correct answer |
| 3 | Belebele | `facebook/belebele` (`yor_Latn`, `eng_Latn`) | test | 900 | Reading passages with a question and its correct answer |
| 4 | MAFAND-MT | `masakhane-io/lafand-mt` on GitHub (`en-yor`) | train | 6,552 | Yorùbá–English sentence pairs from the news (vocabulary and translation) |
| 5 | GSM8K | `openai/gsm8k` (`main`) | train | 7,473 | English math problems with step-by-step solutions |
| | **Total** | | | **15,791** | |

In [36]:
import ast
import io
import re
from datasets import load_dataset
from huggingface_hub import get_session
import pandas as pd

## 1. AfriMGSM: math word problems

Grade-school math problems in Yorùbá and English. The two versions are in the same order, so rows are paired by index (`answer_number` matches in all 258 rows). The 8 train rows have a step-by-step solution; the 250 test rows only have the final number.

In [37]:
data_english_math = load_dataset('masakhane/afrimgsm', 'eng') 
data_yor_math = load_dataset('masakhane/afrimgsm', "yor") 

In [38]:
data_english_math, data_yor_math

(DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }))

In [39]:
data_yor_math["test"]["equation_solution"]

Column([None, None, None, None, None, ...])

In [40]:
data_yor_math["train"]["answer"][0]

'Ìdáhùn lẹ́sẹsẹ: Roger bẹ̀rẹ̀ pẹ̀lú bọ́ọ́lù 5. Agolo 2 bọ́ọ́lù aláfajọ̀gbá kọ̀ọ̀kan 3 jẹ́ bọ́ọ́lù aláfajọ̀gbá 6. 5 + 6 = 11. Ìdáhùn náà ni 11.'

In [41]:
rag_rows = []

In [42]:


for split in ["train", "test"]:
    number_rows = len(data_english_math[split]["question"])
    
    for index in range(number_rows):
        q_yor = data_yor_math[split]["question"][index]
        q_english = data_english_math[split]["question"][index]
        
        ans_yor = data_yor_math[split]["answer"][index]
        
        ans_num = data_yor_math[split]["answer_number"][index]
        eq_english = data_yor_math[split]["equation_solution"][index]
        
        
        context = f"{q_yor} | {q_english}"
        
        
        if ans_yor is not None and eq_english is not None:
            answer = (
                f"[Verified Translation]: {q_english}\n"
                f"[Yorùbá Step-by-Step]: {ans_yor}\n"
                f"[Math Equation]: {eq_english}\n"
                f"[Exact Final Answer]: {ans_num}"
            )
        
        else:
            answer = (
                f"[Verified Parallel Math Problem (Yorùbá -> English)]:\n"
                f"- Yorùbá Text: {q_yor}\n"
                f"- Exact English Meaning: {q_english}\n"
                f"- Verified Numerical Result: {ans_num}"
            )
        
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"afrimgsm_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })



## 2. AfriMMLU: multiple-choice questions

Multiple-choice questions in five subjects: elementary mathematics, high school geography, international law, global facts and high school microeconomics. Each row keeps the four options and the correct answer, in Yorùbá and English.


In [43]:
data_multiple_questions_eng = load_dataset('masakhane/afrimmlu', 'eng')
data_multiple_questions_yor = load_dataset('masakhane/afrimmlu', 'yor')

In [44]:
data_multiple_questions_eng 

DatasetDict({
    validation: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 83
    })
    dev: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 25
    })
    test: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 500
    })
})

### Check for missing values

Counts `None` values in each English split (the Yorùbá splits have none either).

In [45]:
print(data_multiple_questions_eng["validation"]["subject"].count(None) , data_multiple_questions_eng["dev"]["subject"].count(None) , data_multiple_questions_eng["test"]["subject"].count(None))
print(data_multiple_questions_eng["validation"]["question"].count(None) , data_multiple_questions_eng["dev"]["question"].count(None) , data_multiple_questions_eng["test"]["question"].count(None))
print(data_multiple_questions_eng["validation"]["choices"].count(None) , data_multiple_questions_eng["dev"]["choices"].count(None) , data_multiple_questions_eng["test"]["choices"].count(None))

0 0 0
0 0 0
0 0 0


In [46]:
for split in ["validation", "dev", "test"]:
    number_rows = len(data_multiple_questions_eng[split]["question"])
    for index in range(number_rows):
        q_yor = data_multiple_questions_yor[split]["question"][index]
        q_english = data_multiple_questions_eng[split]["question"][index]
        
        eng_sub = data_multiple_questions_eng[split]["subject"][index]
        
        yor_answer = data_multiple_questions_yor[split]["answer"][index]
        eng_anwser = data_multiple_questions_eng[split]["answer"][index]
        
        assert data_multiple_questions_eng[split]["subject"][index] == data_multiple_questions_yor[split]["subject"][index], "Must be the same"
        assert data_multiple_questions_eng[split]["answer"][index] == data_multiple_questions_yor[split]["answer"][index], "Must be the same"
        
        # choices comes as a string, e.g. "['p = 4', 'p = 8', 'p = 12', 'p = 24']"
        choices_yor = ast.literal_eval(data_multiple_questions_yor[split]["choices"][index])
        choices_eng = ast.literal_eval(data_multiple_questions_eng[split]["choices"][index])
        answer_index = "ABCD".index(yor_answer)
        
        options_yor = " | ".join(f"{letter}) {choice}" for letter, choice in zip("ABCD", choices_yor))
        options_eng = " | ".join(f"{letter}) {choice}" for letter, choice in zip("ABCD", choices_eng))
                
        context = f"{q_yor} | {q_english} | {eng_sub}"
        
        answer =  (
                f"[Subject: {eng_sub} | Academic Bilingue Reference]:\n"
                f"- Yorùbá Question: {q_yor}\n"
                f"- English Question: {q_english}\n"
                f"- Options (Yorùbá): {options_yor}\n"
                f"- Options (English): {options_eng}\n"
                f"- Verified Answer (Yorùbá): {yor_answer}) {choices_yor[answer_index]}\n"
                f"- Verified Answer (English): {eng_anwser}) {choices_eng[answer_index]}"
            )
        
        rag_rows.append({
            "language": "yor",
                "source_dataset": f"afrimmlu_{split}",
                "search_context": context,
                "pedagogical_answer": answer
            })

## 3. Belebele: reading comprehension

Short reading passages, each with one or two questions and four possible answers.

In [47]:
data_reading_comphresion_yor = load_dataset('facebook/belebele', "yor_Latn")
data_reading_comphresion_english = load_dataset('facebook/belebele', "eng_Latn")

In [48]:
data_reading_comphresion_yor["test"]["correct_answer_num"]

Column(['1', '1', '2', '2', '3', ...])

In [49]:
data_reading_comphresion_english

DatasetDict({
    test: Dataset({
        features: ['link', 'question_number', 'flores_passage', 'question', 'mc_answer1', 'mc_answer2', 'mc_answer3', 'mc_answer4', 'correct_answer_num', 'dialect', 'ds'],
        num_rows: 900
    })
})

In [50]:
for split in ["test"]:
    # 1. Indexamos el split de inglés por su clave única: (link, question_number)
    eng_lookup = {}
    for i in range(len(data_reading_comphresion_english[split]["question"])):
        key = (
            data_reading_comphresion_english[split]["link"][i],
            data_reading_comphresion_english[split]["question_number"][i]
        )
        eng_lookup[key] = i

    
    number_rows = len(data_reading_comphresion_yor[split]["question"])
    for index in range(number_rows):
        key = (
            data_reading_comphresion_yor[split]["link"][index],
            data_reading_comphresion_yor[split]["question_number"][index]
        )
        
        if key not in eng_lookup:
            continue
            
        eng_idx = eng_lookup[key]
        
        
        yor_q = data_reading_comphresion_yor[split]["question"][index]
        eng_q = data_reading_comphresion_english[split]["question"][eng_idx]
        
        yor_passage = data_reading_comphresion_yor[split]["flores_passage"][index]
        eng_passage = data_reading_comphresion_english[split]["flores_passage"][eng_idx]
        
        
        correct_answer_yor = str(data_reading_comphresion_yor[split]["correct_answer_num"][index]).strip()
        correct_answer_eng = str(data_reading_comphresion_english[split]["correct_answer_num"][eng_idx]).strip()
        
        mac_selection_yor = data_reading_comphresion_yor[split][f"mc_answer{correct_answer_yor}"][index]
        mac_selection_eng = data_reading_comphresion_english[split][f"mc_answer{correct_answer_eng}"][eng_idx]
        
        context = f"{yor_q} | {eng_q}"
        
        answer = (
            f"[Reading Passage (Yorùbá)]: {yor_passage}\n"
            f"[Reading Passage (English)]: {eng_passage}\n"
            f"[Verified Answer (Yorùbá)]: {mac_selection_yor}\n"
            f"[Verified Answer (English)]: {mac_selection_eng}"
        )
    
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"belebele_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })

## 4. MAFAND-MT: Yorùbá–English sentence pairs

Translated news sentences, which give Gemma Yorùbá vocabulary and show how sentences translate. 

In [ ]:
mafand_url = "https://raw.githubusercontent.com/masakhane-io/lafand-mt/main/data/json_files/en-yor/train.json"
response = get_session().get(mafand_url)
response.raise_for_status()
data_translation_mafand = pd.read_json(io.StringIO(response.text), lines=True)

In [52]:
data_translation_mafand

,translation
0,{'en': 'Local radio station in Russia cancels ...
1,{'en': 'Activists in Madrid protest LGBT right...
2,"{'en': 'Echo of Moscow in Yaroslavl, a local a..."
3,"{'en': 'Translation Original Quote', 'yor': 'Ì..."
4,{'en': 'Yesterday we received threats against ...
...,...
6639,"{'en': 'For example, the magazine notes that m..."
6640,"{'en': '“Everybody is happy here,” the article..."
6641,"{'en': 'Brother Stephen Morris, who serves on ..."
6642,{'en': '“While our aim in building a new branc...


In [53]:
seen_pairs = set()
for index in range(len(data_translation_mafand)):
    yor_sentence = data_translation_mafand["translation"][index]["yor"]
    eng_sentence = data_translation_mafand["translation"][index]["en"]
    
    # skip repeated pairs and sentences that were left untranslated (Yorùbá == English)
    if (yor_sentence, eng_sentence) in seen_pairs or yor_sentence.strip() == eng_sentence.strip():
        continue
    seen_pairs.add((yor_sentence, eng_sentence))
    
    context = f"{yor_sentence} | {eng_sentence}"
    
    answer = (
        f"[Verified Translation (Yorùbá -> English)]:\n"
        f"- Yorùbá Sentence: {yor_sentence}\n"
        f"- English Sentence: {eng_sentence}"
    )
    
    rag_rows.append({
        "language": "yor",
        "source_dataset": "mafand_train",
        "search_context": context,
        "pedagogical_answer": answer
    })

## 5. GSM8K: worked math solutions (English)

English math word problems with step-by-step solutions

In [54]:
data_math_gsm8k = load_dataset('openai/gsm8k', 'main')

In [ ]:

for split in ["train"]:
    number_rows = len(data_math_gsm8k[split]["question"])
    for index in range(number_rows):
        q_english = data_math_gsm8k[split]["question"][index]
        full_answer = data_math_gsm8k[split]["answer"][index]
        

        steps_english, ans_num = full_answer.split("####")
        steps_english = re.sub(r"<<[^>]*>>", "", steps_english).strip()
        ans_num = ans_num.strip()
        
        context = q_english
        
        answer = (
            f"[Worked Math Example (English)]: {q_english}\n"
            f"[English Step-by-Step]: {steps_english}\n"
            f"[Exact Final Answer]: {ans_num}"
        )
        
        rag_rows.append({
            "language": "eng",
            "source_dataset": f"gsm8k_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })

## Build and save the corpus

Puts all rows into one DataFrame, prints how many rows come from each dataset and split, and writes `dataset_rag.csv`.

In [56]:
df_african = pd.DataFrame(rag_rows)

In [57]:
print(df_african["source_dataset"].value_counts())
df_african.to_csv("dataset_rag.csv", index=False)

source_dataset
gsm8k_train            7473
mafand_train           6552
belebele_test           900
afrimmlu_test           500
afrimgsm_test           250
afrimmlu_validation      83
afrimmlu_dev             25
afrimgsm_train            8
Name: count, dtype: int64
